# Canvas project group 34: Milestone 2, MovieLens 1M

Thanh Phong Le, Myra Puruganan and Sandra Palomo.

This notebook only orchestrates: every computation is a pure, tested function in `scripts/wrangling.py`, the raw files are only read, and no step is random, so there is no seed to set and every run reproduces the same numbers.

In [ ]:
import json
import os
import sys
from pathlib import Path

In [ ]:
# THE MODULES ARE THE LOGIC: every computation below is a tested function in scripts/.
sys.path.insert(0, str(Path.cwd().parent / "scripts"))

In [ ]:
import matplotlib.pyplot as plt

import wrangling

In [ ]:
DATA = Path(os.environ["MS2_DATA"])
ARCHIVE = Path(os.environ["MS2_ARCHIVE"])
RESULTS = Path(os.environ["MS2_RESULTS"])
TRAINING_FRACTION = 0.8
results: dict[str, object] = {}
figures: list[dict[str, str]] = []

## Getting the data

The data is MovieLens 1M from GroupLens, https://files.grouplens.org/datasets/movielens/ml-1m.zip. The repository's `data:fetch` task downloads it, verifies its SHA-256 against the reviewed pin in `contracts/data-pin.toml` and unpacks it; this notebook reads that verified copy and never writes to it. The archive's size is reported in megabytes, a megabyte being one million bytes.

In [ ]:
results["size_megabytes"] = wrangling.size_megabytes(ARCHIVE)
ratings = wrangling.read_ratings(DATA / "ratings.dat")
users = wrangling.read_users(DATA / "users.dat")
movies = wrangling.read_movies(DATA / "movies.dat")
print(f"Archive size: {results['size_megabytes']:.2f} MB")

## Understanding the data

One row of `ratings.dat` is one rating: one user's star rating, from 1 to 5, of one movie at one time, in Unix seconds. One row of `users.dat` is one user: gender, an age bucket code, an occupation code and a zip code. One row of `movies.dat` is one movie: its title and its pipe-separated genres. Every column is an integer except gender, zip code, title and genres, which are text. Below are each table's shape and each numerical column's count, mean, minimum and maximum.

In [ ]:
results["tables"] = {
    "ratings": [len(ratings), 4],
    "users": [len(users), 5],
    "movies": [len(movies), 3],
}
results["summaries"] = {
    "rating": vars(wrangling.summarize(tuple(r.rating for r in ratings))),
    "timestamp": vars(wrangling.summarize(tuple(r.timestamp for r in ratings))),
    "age": vars(wrangling.summarize(tuple(u.age for u in users))),
    "occupation": vars(wrangling.summarize(tuple(u.occupation for u in users))),
}
print("Shape (rows, columns):", results["tables"])
print("Count, mean, minimum and maximum:", results["summaries"])

## Cleaning and preprocessing

Each check below prints its finding from the data and states the decision taken. The ratings are also judged by the `ratings-quality/v1` contract: validity, uniqueness, referential integrity and completeness, each pass, fail or unknown with the count it found.

### Missing data

Empty fields are counted per column on the raw text, before any field is parsed, and the users with a gap are counted per gender to see whether missingness follows a group.

Decision: nothing is imputed. A rating row with an empty field cannot be used and is excluded and counted; an empty demographic field stays empty and is treated as unknown, because the MovieLens README says the demographics were given voluntarily and not checked.

In [ ]:
columns = {
    "ratings.dat": ("user_id", "movie_id", "rating", "timestamp"),
    "users.dat": wrangling.USER_COLUMNS,
    "movies.dat": ("movie_id", "title", "genres"),
}
results["missing_by_column"] = {
    name: wrangling.missing_by_column(DATA / name, names) for name, names in columns.items()
}
results["missing_by_gender"] = wrangling.missing_by_group(
    DATA / "users.dat", wrangling.USER_COLUMNS, "gender"
)
print("Missing data finding:", results["missing_by_column"], results["missing_by_gender"])
gaps = results["missing_by_column"]["users.dat"]
fig, ax = plt.subplots()
ax.bar(list(gaps), list(gaps.values()))
ax.set_title("Empty fields per users.dat column")
ax.set_xlabel("Column")
ax.set_ylabel("Empty fields")
figures.append({"title": ax.get_title(), "xlabel": ax.get_xlabel(), "ylabel": ax.get_ylabel()})
plt.show()

### Unusual values

Ratings are checked against the 1 to 5 scale and against the users and movies tables; demographic values against the domains the README documents: gender M or F, seven age codes, occupations 0 to 20, and five-digit zip codes.

Decision: a rating outside 1 to 5, a repeated user and movie pair, or a rating whose user or movie is unknown is an error, excluded and counted. A zip code that is not five digits, such as a ZIP+4 or a foreign code, is a real self-reported value: it is kept as text and not used as a numerical feature.

In [ ]:
quality = wrangling.judge(ratings, users, movies)
results["quality"] = [vars(rule) for rule in quality.rules]
results["unusual_users"] = wrangling.unusual_users(users)
print("Unusual values finding:", results["unusual_users"], results["quality"])

### Class balance

The response is the star rating, so each of its five classes is counted, including any that is empty.

Decision: the rating is treated as a number on the 1 to 5 scale and evaluated by root-mean-square error; if the low stars are rare, Milestone 3 also reports the error per star, so that a model cannot look good by predicting only the common stars.

In [ ]:
results["rating_balance"] = wrangling.rating_balance(ratings)
balance = results["rating_balance"]
print("Class balance finding:", balance)
fig, ax = plt.subplots()
ax.bar([str(star) for star in balance], list(balance.values()))
ax.set_title("Ratings per star")
ax.set_xlabel("Star rating")
ax.set_ylabel("Number of ratings")
figures.append({"title": ax.get_title(), "xlabel": ax.get_xlabel(), "ylabel": ax.get_ylabel()})
plt.show()

### Scale of the features

Each numerical column's minimum and maximum are printed below.

Decision: age and occupation are codes for categories, not quantities, so they are one-hot encoded rather than scaled; any numerical feature built in Milestone 3 is standardized with the training set's mean and standard deviation only, so that no scale is learned from test data.

In [ ]:
ranges = {name: (s["minimum"], s["maximum"]) for name, s in results["summaries"].items()}
print("Scale of the features finding:", ranges)

### Information leakage

The ratings are split at one absolute time point, the timestamp below which the training fraction (80 percent) of ratings fall, before any feature is computed. A user's mean over all ratings is refused, because it would use ratings recorded after the ones it is meant to predict.

Decision: every feature is computed from training ratings only; no column is recorded after the rating it would predict, and the timestamp is used only to order and split the ratings.

In [ ]:
cutoff = wrangling.time_cutoff(ratings, TRAINING_FRACTION)
split = wrangling.split_at(ratings, cutoff)
try:
    wrangling.user_means(ratings, cutoff)
    refused = False
except wrangling.LeakageError:
    refused = True
results["leakage_refused"] = refused
results["split"] = {
    "cutoff": cutoff,
    "train": len(split.train),
    "test": len(split.test),
    "train_latest": max(r.timestamp for r in split.train),
    "test_earliest": min(r.timestamp for r in split.test),
}
print("Information leakage finding:", results["split"], "all-ratings mean refused:", refused)
fig, ax = plt.subplots()
ax.hist([r.timestamp for r in ratings], bins=50)
ax.axvline(cutoff, color="black")
ax.set_title("Ratings over time, split at one time point")
ax.set_xlabel("Rating time (Unix seconds)")
ax.set_ylabel("Number of ratings")
figures.append({"title": ax.get_title(), "xlabel": ax.get_xlabel(), "ylabel": ax.get_ylabel()})
plt.show()

## Project question

Revised question: using only the ratings recorded before one time point, the 80th percentile of rating times, can a model predict the star rating of each later rating with a lower root-mean-square error than the baseline that predicts the training ratings' mean? At the end of the project the question is answered by comparing the two errors on the later ratings.

In [ ]:
results["figures"] = figures
RESULTS.write_text(json.dumps(results, indent=2, default=str), encoding="utf-8")
print(f"Results recorded in {RESULTS}")